In [ ]:
# ============================================================
# CELL 1: ENVIRONMENT SETUP
# ============================================================

import os
import random
import pickle
import json
import re
import html
import io

import numpy as np
import pandas as pd

import torch
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from collections import Counter

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix
)

from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_

from google.colab import drive

drive.mount('/content/drive')

# ============================================================
# PATHS
# ============================================================

BASE = '/content/drive/MyDrive/SeaBERT_Final'
IMG_ROOT = BASE

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

for d in [
    'data/embeddings',
    'checkpoints',
    'plots',
    'results'
]:
    os.makedirs(
        os.path.join(BASE, d),
        exist_ok=True
    )

# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True


set_seed(42)

USE_AMP = DEVICE.type == "cuda"

print("Device:", DEVICE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda


In [ ]:
# ============================================================
# CELL 2: TEXT CLEANING
# ============================================================

def clean_text(text):

    text = str(text)

    text = html.unescape(text)

    text = re.sub(
        r'http\S+|www\.\S+',
        '',
        text
    )

    text = re.sub(
        r'@\w+',
        '',
        text
    )

    text = re.sub(
        r'#(\w+)',
        r'\1',
        text
    )

    text = re.sub(
        r'[^\x00-\x7F]+',
        ' ',
        text
    )

    text = re.sub(
        r'<[^>]+>',
        ' ',
        text
    )

    text = re.sub(
        r'([!?.,])\1+',
        r'\1',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text


def clean_dataframe(
    df,
    col='description'
):

    before_n = len(df)

    df = df.copy()

    df[col] = (
        df[col]
        .astype(str)
        .apply(clean_text)
    )

    empty_mask = (
        df[col]
        .str.strip()
        .str.len()
        .eq(0)
    )

    if empty_mask.sum() > 0:

        print(
            f"Dropping {empty_mask.sum()} "
            f"empty text rows"
        )

        df = df[
            ~empty_mask
        ].reset_index(drop=True)

    print(
        f"{before_n} -> {len(df)} rows retained"
    )

    return df


# ============================================================
# LOAD DATA
# ============================================================

df_train = pd.read_csv(
    f'{BASE}/data/processed/damage_train.csv'
)

df_dev = pd.read_csv(
    f'{BASE}/data/processed/damage_dev.csv'
)

df_test = pd.read_csv(
    f'{BASE}/data/processed/damage_test.csv'
)


df_train = clean_dataframe(
    df_train,
    'description'
)

df_dev = clean_dataframe(
    df_dev,
    'description'
)

df_test = clean_dataframe(
    df_test,
    'description'
)


print(
    "\nTrain distribution:"
)

print(
    Counter(
        df_train['severity']
    )
)

print(
    "\nDev distribution:"
)

print(
    Counter(
        df_dev['severity']
    )
)

print(
    "\nTest distribution:"
)

print(
    Counter(
        df_test['severity']
    )
)

2468 -> 2468 rows retained
529 -> 529 rows retained
529 -> 529 rows retained

Train distribution:
Counter({2: 1548, 1: 587, 0: 333})

Dev distribution:
Counter({2: 332, 1: 126, 0: 71})

Test distribution:
Counter({2: 332, 1: 126, 0: 71})


In [ ]:
# ============================================================
# CELL 3: FROZEN FEATURE EXTRACTION
# ============================================================

import torchvision.transforms as T

from PIL import (
    Image,
    ImageFilter,
    ImageEnhance
)

from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

from transformers import (
    AutoTokenizer,
    AutoModel
)


MODEL_NAME = (
    "huawei-noah/"
    "TinyBERT_General_4L_312D"
)

TXT_DIM = 312


# ============================================================
# IMAGE MODEL
# ============================================================

image_model = efficientnet_b0(
    weights=EfficientNet_B0_Weights.IMAGENET1K_V1
).features.to(DEVICE).eval()

for p in image_model.parameters():
    p.requires_grad = False


# ============================================================
# TEXT MODEL
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

text_model = AutoModel.from_pretrained(
    MODEL_NAME
).to(DEVICE).eval()

for p in text_model.parameters():
    p.requires_grad = False


# ============================================================
# IMAGE TRANSFORM
# ============================================================

image_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),

    T.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


MAX_TEXT_LENGTH = 64


# ============================================================
# EVENT EXTRACTION
# ============================================================

def extract_event(filename):

    parts = str(filename).split('/')

    if len(parts) >= 2:
        return parts[1]

    return 'unknown'

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# ============================================================
# CELL 4: MINORITY CLASS IMAGE AUGMENTATION
# ============================================================

import torchvision.transforms as T
from PIL import ImageFilter, ImageEnhance
import io
import random as pyrandom


# ------------------------------------------------------------
# Target distribution
# ------------------------------------------------------------

AUGMENT_TARGETS = {
    0: 900,
    1: 900
}


# ============================================================
# SYNTHETIC HAZE
# ============================================================

class SyntheticHaze:

    def __init__(
        self,
        alpha_range=(0.0, 0.3)
    ):

        self.alpha_range = alpha_range

    def __call__(self, img):

        alpha = pyrandom.uniform(
            *self.alpha_range
        )

        haze_color = pyrandom.choice([
            (200, 200, 200),
            (180, 175, 165),
            (150, 150, 150)
        ])

        haze_layer = Image.new(
            'RGB',
            img.size,
            haze_color
        )

        return Image.blend(
            img,
            haze_layer,
            alpha
        )


# ============================================================
# MOTION BLUR
# ============================================================

class MotionBlur:

    def __init__(
        self,
        radius_range=(0.5, 2.5),
        p=0.35
    ):

        self.radius_range = radius_range
        self.p = p

    def __call__(self, img):

        if pyrandom.random() < self.p:

            return img.filter(
                ImageFilter.GaussianBlur(
                    radius=pyrandom.uniform(
                        *self.radius_range
                    )
                )
            )

        return img


# ============================================================
# JPEG ARTIFACT
# ============================================================

class JPEGCompressionArtifact:

    def __init__(
        self,
        quality_range=(35, 70),
        p=0.5
    ):

        self.quality_range = quality_range
        self.p = p

    def __call__(self, img):

        if pyrandom.random() < self.p:

            buffer = io.BytesIO()

            img.save(
                buffer,
                format='JPEG',
                quality=pyrandom.randint(
                    *self.quality_range
                )
            )

            buffer.seek(0)

            return Image.open(
                buffer
            ).convert('RGB')

        return img


# ============================================================
# EXPOSURE
# ============================================================

class RandomExposure:

    def __init__(
        self,
        brightness_range=(0.6, 1.4),
        contrast_range=(0.7, 1.3)
    ):

        self.brightness_range = brightness_range
        self.contrast_range = contrast_range

    def __call__(self, img):

        img = ImageEnhance.Brightness(
            img
        ).enhance(
            pyrandom.uniform(
                *self.brightness_range
            )
        )

        img = ImageEnhance.Contrast(
            img
        ).enhance(
            pyrandom.uniform(
                *self.contrast_range
            )
        )

        return img


# ============================================================
# AUGMENTATION PIPELINE
# ============================================================

disaster_augment = T.Compose([

    T.RandomHorizontalFlip(
        p=0.5
    ),

    T.RandomRotation(
        degrees=12
    ),

    T.RandomPerspective(
        distortion_scale=0.2,
        p=0.3
    ),

    RandomExposure(),

    T.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.3,
        hue=0.05
    ),

    SyntheticHaze(),

    MotionBlur(),

    JPEGCompressionArtifact(),

    T.RandomAdjustSharpness(
        sharpness_factor=1.8,
        p=0.3
    )
])


# ============================================================
# AUGMENT MINORITY CLASSES
# ============================================================

def augment_minority_rows(
    df,
    targets,
    out_dir
):

    os.makedirs(
        out_dir,
        exist_ok=True
    )

    aug_rows = []

    for cls, target_count in targets.items():

        cls_df = df[
            df['severity'] == cls
        ]

        n_needed = (
            target_count
            - len(cls_df)
        )

        if n_needed <= 0:
            continue

        print(
            f"Class {cls}: "
            f"{len(cls_df)} -> "
            f"{target_count}"
        )

        for i in range(n_needed):

            src_row = cls_df.sample(
                1,
                random_state=5000 + i
            ).iloc[0]

            img = Image.open(
                os.path.join(
                    IMG_ROOT,
                    src_row['filename']
                )
            ).convert('RGB')

            aug_img = disaster_augment(
                img
            )

            original_event = extract_event(
                src_row['filename']
            )

            aug_filename = (
                f"{original_event}/"
                f"aug_bag_{cls}_{i}_"
                f"{os.path.basename(src_row['filename'])}"
            )

            full_save_path = os.path.join(
                out_dir,
                aug_filename
            )

            os.makedirs(
                os.path.dirname(
                    full_save_path
                ),
                exist_ok=True
            )

            aug_img.save(
                full_save_path
            )

            new_row = src_row.copy()

            new_row['filename'] = os.path.join(
                os.path.basename(out_dir),
                aug_filename
            )

            aug_rows.append(
                new_row
            )

    if aug_rows:

        return pd.concat(
            [
                df,
                pd.DataFrame(
                    aug_rows
                )
            ],
            ignore_index=True
        )

    return df


# ============================================================
# CREATE AUGMENTED TRAINING POOL
# ============================================================

df_train_aug = augment_minority_rows(
    df_train,
    AUGMENT_TARGETS,
    out_dir=f"{IMG_ROOT}/augmented_bagging"
)


print(
    "\nOriginal:",
    len(df_train)
)

print(
    "Augmented pool:",
    len(df_train_aug)
)

print(
    "New distribution:",
    Counter(
        df_train_aug['severity']
    )
)

Class 0: 333 -> 900
Class 1: 587 -> 900

Original: 2468
Augmented pool: 3348
New distribution: Counter({2: 1548, 0: 900, 1: 900})


In [ ]:
# ============================================================
# CELL 5: DATASET AND FEATURE EXTRACTION
# ============================================================

class RawImageTextDataset(Dataset):

    def __init__(
        self,
        dataframe
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
        )

    def __len__(self):

        return len(self.df)

    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[index]

        image_path = os.path.join(
            IMG_ROOT,
            row['filename']
        )

        image = Image.open(
            image_path
        ).convert('RGB')

        image = image_transform(
            image
        )

        text = str(
            row['description']
        )

        encoded = tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=MAX_TEXT_LENGTH,
            return_tensors='pt'
        )

        input_ids = (
            encoded['input_ids']
            .squeeze(0)
        )

        attention_mask = (
            encoded['attention_mask']
            .squeeze(0)
        )

        label = int(
            row['severity']
        )

        event = extract_event(
            row['filename']
        )

        return (
            image,
            input_ids,
            attention_mask,
            label,
            event
        )


# ============================================================
# FEATURE EXTRACTION
# ============================================================

def extract_and_cache(
    dataframe,
    cache_path
):

    if os.path.exists(
        cache_path
    ):

        print(
            "Loading:",
            cache_path
        )

        with open(
            cache_path,
            'rb'
        ) as file:

            return pickle.load(
                file
            )

    dataset = RawImageTextDataset(
        dataframe
    )

    loader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    image_features = []
    text_features = []
    text_masks = []
    labels = []
    events = []

    with torch.no_grad():

        for (
            images,
            input_ids,
            attn_masks,
            batch_labels,
            batch_events
        ) in tqdm(
            loader,
            desc=(
                "Extracting "
                + os.path.basename(
                    cache_path
                )
            )
        ):

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            input_ids = input_ids.to(
                DEVICE,
                non_blocking=True
            )

            attn_masks = attn_masks.to(
                DEVICE,
                non_blocking=True
            )

            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=USE_AMP
            ):

                img_feat = image_model(
                    images
                )

                txt_out = text_model(
                    input_ids=input_ids,
                    attention_mask=attn_masks
                ).last_hidden_state

            image_features.append(
                img_feat.cpu()
                .numpy()
                .astype(np.float16)
            )

            text_features.append(
                txt_out.cpu()
                .numpy()
                .astype(np.float16)
            )

            text_masks.append(
                attn_masks.cpu()
                .numpy()
                .astype(np.int64)
            )

            labels.extend(
                batch_labels
            )

            events.extend(
                batch_events
            )

    data = {

        'img_maps':
            np.concatenate(
                image_features,
                axis=0
            ),

        'txt_seqs':
            np.concatenate(
                text_features,
                axis=0
            ),

        'txt_masks':
            np.concatenate(
                text_masks,
                axis=0
            ),

        'labels':
            np.array(
                labels,
                dtype=np.int64
            ),

        'events':
            events
    }

    with open(
        cache_path,
        'wb'
    ) as file:

        pickle.dump(
            data,
            file
        )

    print(
        f"Saved {cache_path}: "
        f"{os.path.getsize(cache_path)/1024**2:.2f} MB"
    )

    return data

In [ ]:
# ============================================================
# CELL 6: EXTRACT OR LOAD FEATURES
# ============================================================

train_data = extract_and_cache(
    df_train_aug,
    f'{BASE}/data/embeddings/'
    f'severity_train_aug_bagging.pkl'
)

dev_data = extract_and_cache(
    df_dev,
    f'{BASE}/data/embeddings/'
    f'severity_dev_clean.pkl'
)

test_data = extract_and_cache(
    df_test,
    f'{BASE}/data/embeddings/'
    f'severity_test_clean.pkl'
)


print(
    "\nDataset sizes:"
)

print(
    "Train:",
    len(train_data['labels'])
)

print(
    "Dev:",
    len(dev_data['labels'])
)

print(
    "Test:",
    len(test_data['labels'])
)

Loading: /content/drive/MyDrive/SeaBERT_Final/data/embeddings/severity_train_aug_bagging.pkl
Loading: /content/drive/MyDrive/SeaBERT_Final/data/embeddings/severity_dev_clean.pkl
Loading: /content/drive/MyDrive/SeaBERT_Final/data/embeddings/severity_test_clean.pkl

Dataset sizes:
Train: 3348
Dev: 529
Test: 529


In [ ]:
# ============================================================
# CELL 7: DATASET AND DATALOADERS
# ============================================================

class SeverityDataset(Dataset):

    def __init__(
        self,
        data
    ):

        self.img_maps = torch.from_numpy(
            data['img_maps']
        ).float()

        self.txt_seqs = torch.from_numpy(
            data['txt_seqs']
        ).float()

        self.txt_masks = torch.from_numpy(
            data['txt_masks']
        ).float()

        self.labels = torch.from_numpy(
            data['labels']
        ).long()

        # Event IDs
        self.events_raw = data['events']

    def __len__(self):

        return len(
            self.labels
        )

    def __getitem__(
        self,
        idx
    ):

        return (
            self.img_maps[idx],
            self.txt_seqs[idx],
            self.txt_masks[idx],
            self.labels[idx],
            self.events_raw[idx]
        )


train_ds = SeverityDataset(
    train_data
)

dev_ds = SeverityDataset(
    dev_data
)

test_ds = SeverityDataset(
    test_data
)


# ============================================================
# CLASS DISTRIBUTION
# ============================================================

train_counts = np.bincount(
    train_data['labels'],
    minlength=3
)

print(
    "Training distribution:",
    train_counts
)


# ============================================================
# CLASS WEIGHTS
#
# Because classes 0 and 1 were augmented to 900,
# the expected distribution is approximately:
#
#       class 0 = 900
#       class 1 = 900
#       class 2 = 1548
#
# ============================================================

counts = train_counts.astype(
    np.float32
)

class_weights = (
    1.0 / np.sqrt(
        np.maximum(
            counts,
            1
        )
    )
)

class_weights = (
    class_weights
    / class_weights.sum()
    * 3
)

SEV_WEIGHTS = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)

print(
    "CORAL class weights:",
    SEV_WEIGHTS.detach()
    .cpu()
    .numpy()
)


# ============================================================
# LOADERS
# ============================================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

dev_loader = DataLoader(
    dev_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

Training distribution: [ 900  900 1548]
CORAL class weights: [1.0859758 1.0859758 0.8280486]


In [ ]:
# ============================================================
# CELL 8: SHARED TEXT ADAPTER
# ============================================================

class AdapterPoolText(nn.Module):

    def __init__(
        self,
        in_dim=TXT_DIM,
        out_dim=256,
        bottleneck=64
    ):

        super().__init__()

        self.down = nn.Linear(
            in_dim,
            bottleneck
        )

        self.up = nn.Linear(
            bottleneck,
            in_dim
        )

        self.ln_adapter = nn.LayerNorm(
            in_dim
        )

        self.act = nn.GELU()

        self.proj = nn.Linear(
            in_dim,
            out_dim
        )

    def forward(
        self,
        txt_seq,
        mask
    ):

        residual = txt_seq

        x = self.act(
            self.down(txt_seq)
        )

        x = self.up(x)

        txt_seq = self.ln_adapter(
            residual + x
        )

        m = mask.unsqueeze(-1)

        pooled = (
            txt_seq * m
        ).sum(dim=1) / m.sum(
            dim=1
        ).clamp(
            min=1e-6
        )

        return self.proj(
            pooled
        )

In [ ]:
# ============================================================
# CELL 9: CORAL HEAD + LOSS
# ============================================================

class CoralHead(nn.Module):

    def __init__(
        self,
        in_dim,
        num_classes=3
    ):

        super().__init__()

        self.num_thresholds = (
            num_classes - 1
        )

        self.fc = nn.Linear(
            in_dim,
            1,
            bias=False
        )

        self.bias0 = nn.Parameter(
            torch.zeros(1)
        )

        self.deltas = nn.Parameter(
            torch.ones(
                max(
                    self.num_thresholds - 1,
                    0
                )
            )
        )

    def forward(
        self,
        x
    ):

        base = self.fc(x)

        biases = [
            self.bias0
        ]

        b = self.bias0

        for i in range(
            self.num_thresholds - 1
        ):

            b = (
                b
                - F.softplus(
                    self.deltas[i]
                )
            )

            biases.append(b)

        biases = torch.cat(
            biases
        )

        return (
            base
            + biases.unsqueeze(0)
        )


class CoralLoss(nn.Module):

    def __init__(
        self,
        num_classes=3,
        class_weights=None
    ):

        super().__init__()

        self.num_classes = num_classes
        self.class_weights = class_weights

    def forward(
        self,
        logits,
        targets
    ):

        thresholds = torch.arange(
            self.num_classes - 1,
            device=logits.device
        )

        ext_targets = (
            targets.unsqueeze(1)
            > thresholds.unsqueeze(0)
        ).float()

        losses = F.binary_cross_entropy_with_logits(
            logits,
            ext_targets,
            reduction="none"
        ).sum(dim=1)

        if self.class_weights is not None:

            losses = (
                losses
                * self.class_weights[
                    targets
                ]
            )

        return losses.mean()


def coral_decode(
    logits
):

    return (
        torch.sigmoid(logits) > 0.5
    ).sum(dim=1)

In [ ]:
# ============================================================
# CELL 10: EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader
):

    model.eval()

    targets = []
    preds = []

    for (
        img,
        txt,
        mask,
        label,
        eid
    ) in loader:

        img = img.to(
            DEVICE,
            non_blocking=True
        )

        txt = txt.to(
            DEVICE,
            non_blocking=True
        )

        mask = mask.to(
            DEVICE,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP
        ):

            logits = model(
                img,
                txt,
                mask
            )

        pred = coral_decode(
            logits
        )

        preds.extend(
            pred.cpu().numpy()
        )

        targets.extend(
            label.numpy()
        )

    targets = np.asarray(
        targets
    )

    preds = np.asarray(
        preds
    )

    macro_f1 = f1_score(
        targets,
        preds,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        targets,
        preds,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(
        targets,
        preds
    )

    mcc = matthews_corrcoef(
        targets,
        preds
    )

    kappa = cohen_kappa_score(
        targets,
        preds
    )

    qwk = cohen_kappa_score(
        targets,
        preds,
        weights="quadratic"
    )

    return (
        macro_f1,
        weighted_f1,
        accuracy,
        mcc,
        kappa,
        qwk,
        targets,
        preds
    )

In [ ]:
# ============================================================
# CELL 11: FUSION ABLATION MODEL
# ============================================================

class FusionAblationMember(nn.Module):

    def __init__(
        self,
        fusion_mode="bi",
        z_dim=64,
        num_classes=3
    ):

        super().__init__()

        assert fusion_mode in [
            "concat",
            "uni",
            "bi"
        ]

        self.fusion_mode = fusion_mode

        # ====================================================
        # VISUAL PROJECTION
        # ====================================================

        self.visual_proj = nn.Sequential(
            nn.Linear(
                1280,
                256
            ),
            nn.LayerNorm(256),
            nn.ReLU()
        )

        # ====================================================
        # TEXT PROJECTION
        # ====================================================

        self.text_pool = AdapterPoolText(
            in_dim=TXT_DIM,
            out_dim=256,
            bottleneck=64
        )

        self.text_proj = nn.Sequential(
            nn.Linear(
                256,
                256
            ),
            nn.LayerNorm(256),
            nn.ReLU()
        )

        # ====================================================
        # V -> T ATTENTION
        # ====================================================

        self.v2t_attn = nn.MultiheadAttention(
            256,
            4,
            dropout=0.1,
            batch_first=True
        )

        # ====================================================
        # T -> V ATTENTION
        # ====================================================

        self.t2v_attn = nn.MultiheadAttention(
            256,
            4,
            dropout=0.1,
            batch_first=True
        )

        self.ln_v = nn.LayerNorm(
            256
        )

        self.ln_t = nn.LayerNorm(
            256
        )

        # ====================================================
        # SELF ATTENTION
        # ====================================================

        self.self_attn = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=4,
            dim_feedforward=512,
            dropout=0.1,
            batch_first=True,
            norm_first=True
        )

        # ====================================================
        # FINAL REPRESENTATION
        # ====================================================

        self.z_proj = nn.Sequential(
            nn.Linear(
                512,
                z_dim
            ),
            nn.LayerNorm(z_dim),
            nn.ReLU()
        )

        # ====================================================
        # CORAL
        # ====================================================

        self.coral_head = CoralHead(
            z_dim,
            num_classes=num_classes
        )

    # ========================================================
    # EMBEDDING
    # ========================================================

    def embed(
        self,
        img_map,
        txt_seq,
        txt_mask
    ):

        # ----------------------------------------------------
        # IMAGE
        # ----------------------------------------------------

        v = self.visual_proj(
            img_map.mean(
                dim=[2, 3]
            )
        )

        # ----------------------------------------------------
        # TEXT
        # ----------------------------------------------------

        t = self.text_proj(
            self.text_pool(
                txt_seq,
                txt_mask
            )
        )

        v_seq = v.unsqueeze(1)
        t_seq = t.unsqueeze(1)

        # ====================================================
        # CONCAT
        # ====================================================

        if self.fusion_mode == "concat":

            fused = torch.cat(
                [
                    v,
                    t
                ],
                dim=-1
            )

        # ====================================================
        # UNIDIRECTIONAL
        #
        # V -> T
        # ====================================================

        elif self.fusion_mode == "uni":

            Q, _ = self.v2t_attn(
                v_seq,
                t_seq,
                t_seq
            )

            v_attn = self.ln_v(
                v + Q.squeeze(1)
            )

            fused = torch.cat(
                [
                    v_attn,
                    t
                ],
                dim=-1
            )

        # ====================================================
        # BIDIRECTIONAL
        #
        # V -> T
        # T -> V
        # ====================================================

        elif self.fusion_mode == "bi":

            # V -> T

            Q, _ = self.v2t_attn(
                v_seq,
                t_seq,
                t_seq
            )

            v_attn = self.ln_v(
                v + Q.squeeze(1)
            )

            # T -> V

            P, _ = self.t2v_attn(
                t_seq,
                v_seq,
                v_seq
            )

            t_attn = self.ln_t(
                t + P.squeeze(1)
            )

            # Two modality tokens

            pair = torch.stack(
                [
                    v_attn,
                    t_attn
                ],
                dim=1
            )

            # Self-attention

            pair_sa = self.self_attn(
                pair
            )

            fused = torch.cat(
                [
                    pair_sa[:, 0, :],
                    pair_sa[:, 1, :]
                ],
                dim=-1
            )

        # ----------------------------------------------------
        # Final latent embedding
        # ----------------------------------------------------

        z = self.z_proj(
            fused
        )

        return z

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(
        self,
        img_map,
        txt_seq,
        txt_mask
    ):

        z = self.embed(
            img_map,
            txt_seq,
            txt_mask
        )

        return self.coral_head(
            z
        )

In [ ]:
# ============================================================
# CELL 12: TRAIN SINGLE-MODEL ABLATIONS
# ============================================================

def train_ablation_model(
    fusion_mode,
    epochs=25,
    patience_limit=6,
    lr=1e-3,
    seed=42
):

    print("\n")
    print("=" * 70)
    print(
        f"TRAINING: {fusion_mode.upper()}"
    )
    print("=" * 70)

    set_seed(seed)

    # --------------------------------------------------------
    # ONE MODEL ONLY
    # --------------------------------------------------------

    model = FusionAblationMember(
        fusion_mode=fusion_mode,
        z_dim=64,
        num_classes=3
    ).to(DEVICE)

    # --------------------------------------------------------
    # SAME DATA FOR ALL ABLATIONS
    # --------------------------------------------------------

    loader = train_loader

    # --------------------------------------------------------
    # CORAL LOSS
    # --------------------------------------------------------

    criterion = CoralLoss(
        num_classes=3,
        class_weights=SEV_WEIGHTS
    )

    # --------------------------------------------------------
    # OPTIMIZER
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    best_f1 = -np.inf
    best_state = None
    patience = 0

    # ========================================================
    # TRAIN
    # ========================================================

    for epoch in range(
        epochs
    ):

        model.train()

        running_loss = 0.0
        batches = 0

        for (
            img,
            txt,
            mask,
            label,
            eid
        ) in loader:

            img = img.to(
                DEVICE,
                non_blocking=True
            )

            txt = txt.to(
                DEVICE,
                non_blocking=True
            )

            mask = mask.to(
                DEVICE,
                non_blocking=True
            )

            label = label.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=USE_AMP
            ):

                logits = model(
                    img,
                    txt,
                    mask
                )

                loss = criterion(
                    logits,
                    label
                )

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            running_loss += (
                loss.item()
            )

            batches += 1

        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        (
            macro_f1,
            weighted_f1,
            accuracy,
            mcc,
            kappa,
            qwk,
            _,
            _
        ) = evaluate(
            model,
            dev_loader
        )

        avg_loss = (
            running_loss
            / max(batches, 1)
        )

        print(
            f"{fusion_mode.upper()} | "
            f"Epoch {epoch+1:02d} | "
            f"Loss: {avg_loss:.4f} | "
            f"Macro-F1: {macro_f1:.4f} | "
            f"Accuracy: {accuracy:.4f}"
        )

        # ----------------------------------------------------
        # BEST MODEL
        # ----------------------------------------------------

        if macro_f1 > best_f1:

            best_f1 = macro_f1

            patience = 0

            best_state = {
                k: v.detach()
                .cpu()
                .clone()
                for k, v in model.state_dict().items()
            }

        else:

            patience += 1

        if patience >= patience_limit:

            print(
                "Early stopping."
            )

            break

    # --------------------------------------------------------
    # RESTORE BEST MODEL
    # --------------------------------------------------------

    if best_state is not None:

        model.load_state_dict(
            best_state
        )

    print(
        f"\nBest {fusion_mode.upper()} "
        f"Dev Macro-F1: {best_f1:.4f}"
    )

    return model

In [ ]:
# ============================================================
# CELL 13: TRAIN THE THREE ABLATIONS
# ============================================================

concat_model = train_ablation_model(
    fusion_mode="concat",
    seed=100
)

uni_model = train_ablation_model(
    fusion_mode="uni",
    seed=200
)

bi_model = train_ablation_model(
    fusion_mode="bi",
    seed=300
)



TRAINING: CONCAT
CONCAT | Epoch 01 | Loss: 0.9588 | Macro-F1: 0.5320 | Accuracy: 0.6711
CONCAT | Epoch 02 | Loss: 0.7453 | Macro-F1: 0.5061 | Accuracy: 0.6843
CONCAT | Epoch 03 | Loss: 0.6100 | Macro-F1: 0.5020 | Accuracy: 0.6446
CONCAT | Epoch 04 | Loss: 0.4988 | Macro-F1: 0.5124 | Accuracy: 0.6484
CONCAT | Epoch 05 | Loss: 0.4231 | Macro-F1: 0.4789 | Accuracy: 0.6578
CONCAT | Epoch 06 | Loss: 0.3661 | Macro-F1: 0.5384 | Accuracy: 0.6673
CONCAT | Epoch 07 | Loss: 0.3297 | Macro-F1: 0.5070 | Accuracy: 0.6522
CONCAT | Epoch 08 | Loss: 0.2845 | Macro-F1: 0.4899 | Accuracy: 0.6446
CONCAT | Epoch 09 | Loss: 0.2714 | Macro-F1: 0.5137 | Accuracy: 0.6692
CONCAT | Epoch 10 | Loss: 0.2465 | Macro-F1: 0.5432 | Accuracy: 0.6427
CONCAT | Epoch 11 | Loss: 0.2237 | Macro-F1: 0.5200 | Accuracy: 0.6749
CONCAT | Epoch 12 | Loss: 0.2044 | Macro-F1: 0.5498 | Accuracy: 0.6673
CONCAT | Epoch 13 | Loss: 0.1870 | Macro-F1: 0.5071 | Accuracy: 0.6673
CONCAT | Epoch 14 | Loss: 0.1843 | Macro-F1: 0.5241 | Accu

In [ ]:
# ============================================================
# CELL 14: ABLATION EVALUATION
# ============================================================

def evaluate_ablation(
    name,
    model
):

    (
        macro_f1,
        weighted_f1,
        accuracy,
        mcc,
        kappa,
        qwk,
        targets,
        preds
    ) = evaluate(
        model,
        test_loader
    )

    cm = confusion_matrix(
        targets,
        preds
    )

    print("\n")
    print("=" * 70)
    print(name)
    print("=" * 70)

    print(
        f"Macro F1        : {macro_f1:.4f}"
    )

    print(
        f"Weighted F1     : {weighted_f1:.4f}"
    )

    print(
        f"Accuracy         : {accuracy:.4f}"
    )

    print(
        f"MCC              : {mcc:.4f}"
    )

    print(
        f"Cohen Kappa      : {kappa:.4f}"
    )

    print(
        f"Quadratic Kappa  : {qwk:.4f}"
    )

    print(
        "\nClassification Report:"
    )

    print(
        classification_report(
            targets,
            preds,
            target_names=[
                "Little/No Damage",
                "Mild Damage",
                "Severe Damage"
            ],
            digits=4,
            zero_division=0
        )
    )

    print(
        "Confusion Matrix:"
    )

    print(cm)

    return {
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "accuracy": float(accuracy),
        "mcc": float(mcc),
        "kappa": float(kappa),
        "qwk": float(qwk)
    }


concat_result = evaluate_ablation(
    "CONCATENATION",
    concat_model
)

uni_result = evaluate_ablation(
    "UNIDIRECTIONAL ATTENTION",
    uni_model
)

bi_result = evaluate_ablation(
    "BIDIRECTIONAL ATTENTION",
    bi_model
)



CONCATENATION
Macro F1        : 0.5113
Weighted F1     : 0.6235
Accuracy         : 0.6257
MCC              : 0.2913
Cohen Kappa      : 0.2912
Quadratic Kappa  : 0.4407

Classification Report:
                  precision    recall  f1-score   support

Little/No Damage     0.4110    0.4225    0.4167        71
     Mild Damage     0.3529    0.3333    0.3429       126
   Severe Damage     0.7685    0.7801    0.7743       332

        accuracy                         0.6257       529
       macro avg     0.5108    0.5120    0.5113       529
    weighted avg     0.6216    0.6257    0.6235       529

Confusion Matrix:
[[ 30  23  18]
 [ 24  42  60]
 [ 19  54 259]]


UNIDIRECTIONAL ATTENTION
Macro F1        : 0.5250
Weighted F1     : 0.6423
Accuracy         : 0.6389
MCC              : 0.3333
Cohen Kappa      : 0.3328
Quadratic Kappa  : 0.4536

Classification Report:
                  precision    recall  f1-score   support

Little/No Damage     0.3563    0.4366    0.3924        71
     Mild D

In [ ]:
# ============================================================
# CELL 15: ABLATION SUMMARY
# ============================================================

ablation_results = pd.DataFrame([

    {
        "Fusion": "Concat",
        "Macro F1": concat_result["macro_f1"],
        "Weighted F1": concat_result["weighted_f1"],
        "Accuracy": concat_result["accuracy"],
        "MCC": concat_result["mcc"],
        "Kappa": concat_result["kappa"],
        "Quadratic Kappa": concat_result["qwk"]
    },

    {
        "Fusion": "Uni-directional",
        "Macro F1": uni_result["macro_f1"],
        "Weighted F1": uni_result["weighted_f1"],
        "Accuracy": uni_result["accuracy"],
        "MCC": uni_result["mcc"],
        "Kappa": uni_result["kappa"],
        "Quadratic Kappa": uni_result["qwk"]
    },

    {
        "Fusion": "Bi-directional",
        "Macro F1": bi_result["macro_f1"],
        "Weighted F1": bi_result["weighted_f1"],
        "Accuracy": bi_result["accuracy"],
        "MCC": bi_result["mcc"],
        "Kappa": bi_result["kappa"],
        "Quadratic Kappa": bi_result["qwk"]
    }
])


print(
    "\n" + "=" * 100
)

print(
    "FUSION ABLATION RESULTS"
)

print(
    "=" * 100
)

print(
    ablation_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


FUSION ABLATION RESULTS
         Fusion  Macro F1  Weighted F1  Accuracy    MCC  Kappa  Quadratic Kappa
         Concat    0.5113       0.6235    0.6257 0.2913 0.2912           0.4407
Uni-directional    0.5250       0.6423    0.6389 0.3333 0.3328           0.4536
 Bi-directional    0.5144       0.6370    0.6352 0.3182 0.3174           0.4390


In [ ]:
import json

# ============================================================
# CELL 16: SAVE ABLATION RESULTS
# ============================================================

results_path = (
    f'{BASE}/results/'
    'fusion_ablation_augmented_900.json'
)


with open(
    results_path,
    'w'
) as file:

    json.dump(
        ablation_results.to_dict('records'), # Convert DataFrame to list of dictionaries
        file,
        indent=2
    )


print(
    "Saved:",
    results_path
)

Saved: /content/drive/MyDrive/SeaBERT_Final/results/fusion_ablation_augmented_900.json


In [ ]:
# ============================================================
# FINAL CELL: BELT TEST EVALUATION
# ============================================================

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score
)

@torch.no_grad()
def evaluate_belt(model, loader):
    model.eval()

    targets = []
    preds = []

    for img, txt, mask, label, eid in loader:

        img = img.to(DEVICE, non_blocking=True)
        txt = txt.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP
        ):
            logits = model(img, txt, mask)

        pred = coral_decode(logits)

        preds.extend(pred.cpu().numpy())
        targets.extend(label.numpy())

    targets = np.array(targets)
    preds = np.array(preds)

    macro_f1 = f1_score(
        targets, preds,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        targets, preds,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(targets, preds)

    mcc = matthews_corrcoef(targets, preds)

    kappa = cohen_kappa_score(targets, preds)

    qwk = cohen_kappa_score(
        targets,
        preds,
        weights="quadratic"
    )

    cm = confusion_matrix(targets, preds)

    return (
        macro_f1,
        weighted_f1,
        accuracy,
        mcc,
        kappa,
        qwk,
        targets,
        preds,
        cm
    )


# ------------------------------------------------------------
# Run BELT on TEST SET
# ------------------------------------------------------------

(
    macro_f1,
    weighted_f1,
    accuracy,
    mcc,
    kappa,
    qwk,
    targets,
    preds,
    cm
) = evaluate_belt(combiner, test_loader)


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("=" * 60)
print("BELT — FINAL TEST RESULTS")
print("=" * 60)

print(f"Macro F1        : {macro_f1:.4f}")
print(f"Weighted F1     : {weighted_f1:.4f}")
print(f"Accuracy        : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"MCC             : {mcc:.4f}")
print(f"Cohen Kappa     : {kappa:.4f}")
print(f"Quadratic Kappa : {qwk:.4f}")

print("\nEnsemble weights:")
print(
    combiner.w.detach()
    .cpu()
    .numpy()
    .round(4)
)

print("\nClassification Report:")
print(
    classification_report(
        targets,
        preds,
        target_names=[
            "Little/No Damage",
            "Mild Damage",
            "Severe Damage"
        ],
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(cm)

print("=" * 60)

NameError: name 'combiner' is not defined